# ALS Baseline: Books + Movies Intersection

Train one implicit ALS model on merged Books and Movies interactions from the prepared Google Drive splits. Recommendations are filtered back to the target domain before MAP@10 evaluation.

In [18]:
!test -d /content/recommendation-system/.git \
  || git clone -b als https://github.com/mkobycheva/recommendation-system.git /content/recommendation-system

%cd /content/recommendation-system
!git fetch origin als
!git checkout als
!git reset --hard origin/als
!git log -1 --oneline
!pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/recommendation-system
From https://github.com/mkobycheva/recommendation-system
 * branch            als        -> FETCH_HEAD
Already on 'als'
Your branch is up to date with 'origin/als'.
HEAD is now at eb85d88 removed unnecessary import from build_matrix.py
eb85d88 (HEAD -> als, origin/als) removed unnecessary import from build_matrix.py
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import implicit
import numpy as np
import pandas as pd

from src.data.build_matrix import add_domain_item_ids, build_implicit_als_matrix
from src.evaluation.metrics import map_at_k, ndcg_at_k, recall_at_k
from src.evaluation.cross_domain_eval import evaluate_multi_domain

## Load Prepared Splits

The split CSVs are expected to contain the Books/Movies intersecting users and time-based train/validation/test rows.

In [20]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [21]:
books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df = pd.concat([books_test, movies_test], ignore_index=True)

train_df[['user_id', 'item_id', 'domain']].head()

,user_id,item_id,domain
0,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0061713244,books
1,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0871139634,books
2,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0385521685,books
3,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0848732634,books
4,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0151014981,books


## Shared Indexes and Train Matrix

In [22]:
interaction_matrix = build_implicit_als_matrix(train_df)

user_item_train = interaction_matrix.user_item_train
user2idx = interaction_matrix.user2idx
item2idx = interaction_matrix.item2idx
idx2item = interaction_matrix.idx2item
item_domain = interaction_matrix.item_domain
train_seen_idx_by_user = interaction_matrix.train_seen_idx_by_user
domain_item_indices = interaction_matrix.domain_item_indices

n_users, n_items = user_item_train.shape

print(f'users={n_users:,}, items={n_items:,}, train_interactions={user_item_train.nnz:,}')
print(train_df['domain'].value_counts())

users=127,188, items=587,064, train_interactions=3,750,813
domain
movies    1887583
books     1863230
Name: count, dtype: int64


In [23]:
assert user_item_train.shape == (n_users, n_items)
assert len(domain_item_indices['books']) > 0
assert len(domain_item_indices['movies']) > 0

## Train Collective ALS

In [26]:
FACTORS = 128          # Increased from 64
REGULARIZATION = 0.05  # Increased from 0.01
ITERATIONS = 20        # Kept at 20
ALPHA = 15             # Lowered from 40
K = 10

als_model = implicit.als.AlternatingLeastSquares(
    factors=FACTORS,
    regularization=REGULARIZATION,
    iterations=ITERATIONS,
    random_state=42,
)

item_user_train = (user_item_train * ALPHA).T.tocsr()
als_model.fit(item_user_train)

# The model is fit on item-user input, so implicit stores item vectors in
# user_factors and user vectors in item_factors.
als_item_factors = als_model.user_factors
als_user_factors = als_model.item_factors

assert als_item_factors.shape[0] == n_items
assert als_user_factors.shape[0] == n_users

  0%|          | 0/20 [00:00<?, ?it/s]

## Domain-Filtered Recommendations

In [30]:
_candidate_to_position = {
    domain: {item_idx: pos for pos, item_idx in enumerate(indices)}
    for domain, indices in domain_item_indices.items()
}

def relevant_items_by_user(split_df, target_domain):
    domain_rows = split_df[split_df['domain'] == target_domain]
    return domain_rows.groupby('user_id')['item_id'].agg(set).to_dict()


def recommend_for_users(user_ids, target_domain, k=10, batch_size=512, max_score_elements=20_000_000):
    candidates = domain_item_indices[target_domain]
    candidate_factors = als_item_factors[candidates]
    candidate_to_position = _candidate_to_position[target_domain]  # O(1) instant reuse

    recommendations = {}
    user_ids = list(user_ids)
    effective_batch_size = max(1, min(batch_size, max_score_elements // max(len(candidates), 1)))

    for batch_start in range(0, len(user_ids), effective_batch_size):
        batch = user_ids[batch_start:batch_start + effective_batch_size]
        valid_user_ids = []
        user_indices = []

        for user_id in batch:
            user_idx = user2idx.get(user_id)
            if user_idx is None:
                recommendations[user_id] = []
                continue
            valid_user_ids.append(user_id)
            user_indices.append(user_idx)

        if not user_indices:
            continue

        user_factors = als_user_factors[np.array(user_indices, dtype=np.int64)]
        all_scores = user_factors @ candidate_factors.T

        for row_idx, user_id in enumerate(valid_user_ids):
            scores = all_scores[row_idx].astype(np.float64, copy=True)
            seen_items = train_seen_idx_by_user.get(user_indices[row_idx], set())

            # Efficiently extract in-domain training items to apply the masking penalty
            known_positions = np.fromiter(
                (candidate_to_position[item_idx] for item_idx in seen_items if item_idx in candidate_to_position),
                dtype=np.int64,
            )
            if known_positions.size:
                scores[known_positions] = -np.inf

            finite_count = np.isfinite(scores).sum()
            if finite_count == 0:
                recommendations[user_id] = []
                continue

            top_n = min(k, int(finite_count))
            top_positions = np.argpartition(-scores, top_n - 1)[:top_n]
            top_positions = top_positions[np.argsort(-scores[top_positions])]
            recommendations[user_id] = [idx2item[int(candidates[pos])] for pos in top_positions]

        if (batch_start // effective_batch_size) % 20 == 0:
            print(f'  Processed {batch_start:,}/{len(user_ids):,} users for {target_domain}...')

    return recommendations

## Evaluate and Save Metrics

In [ ]:
K = 10

# 1. Clear interface router mapping to notebook inference algorithm
def run_als_inference(user_ids, target_domain, k=10):
    return recommend_for_users(user_ids, target_domain, k=k)

# 2. Unified, safe execution sweep across splits simultaneously
valid_results, test_results = evaluate_multi_domain(
    split_dfs={'valid': valid_df, 'test': test_df},
    recommend_func=run_als_inference,
    k=K
)

# 3. Correctly label model payload before logging to CSV
os.makedirs('results', exist_ok=True)
result_row = {
    'model': 'implicit_als_books_movies_joint_weighted',  # Corrected name label
    'factors': FACTORS,
    'regularization': REGULARIZATION,
    'iterations': ITERATIONS,
    'alpha': ALPHA,
    'k': K,
    'n_users': n_users,
    'n_items': n_items,
    'n_train_interactions': user_item_train.nnz,
    'books_valid_ndcg@10': valid_results['books']['ndcg@10'],
    'movies_valid_ndcg@10': valid_results['movies']['ndcg@10'],
    'valid_macro_ndcg@10': round((valid_results['books']['ndcg@10'] + valid_results['movies']['ndcg@10']) / 2, 4),
    'books_valid_recall@10': valid_results['books']['recall@10'],
    'movies_valid_recall@10': valid_results['movies']['recall@10'],
    'valid_macro_recall@10': round((valid_results['books']['recall@10'] + valid_results['movies']['recall@10']) / 2, 4),
    'books_valid_map@10': valid_results['books']['map@10'],
    'movies_valid_map@10': valid_results['movies']['map@10'],
    'valid_macro_map@10': round((valid_results['books']['map@10'] + valid_results['movies']['map@10']) / 2, 4),
    'books_test_ndcg@10': test_results['books']['ndcg@10'],
    'movies_test_ndcg@10': test_results['movies']['ndcg@10'],
    'test_macro_ndcg@10': round((test_results['books']['ndcg@10'] + test_results['movies']['ndcg@10']) / 2, 4),
    'books_test_recall@10': test_results['books']['recall@10'],
    'movies_test_recall@10': test_results['movies']['recall@10'],
    'test_macro_recall@10': round((test_results['books']['recall@10'] + test_results['movies']['recall@10']) / 2, 4),
    'books_test_map@10': test_results['books']['map@10'],
    'movies_test_map@10': test_results['movies']['map@10'],
    'test_macro_map@10': round((test_results['books']['map@10'] + test_results['movies']['map@10']) / 2, 4),
}

results_df = pd.DataFrame([result_row])
results_df.to_csv('results/als_baseline_metrics.csv', index=False)
results_df